# PricePoint — 06 · KNN (K-Nearest Neighbours)

**What it is:** to price a laptop, it finds the **k most similar laptops** in the training data and **averages their prices**. It learns no formula — it simply remembers the data and looks up the closest matches.

**Important:** "similar" means *close in distance*, so KNN is sensitive to scale (`8GB` of RAM vs `2073600` pixels). Like SVR, we add a **StandardScaler** step so every feature counts fairly.

**One setting to choose:** **k**, the number of neighbours. Too small → follows noise. Too large → averages very different laptops. We pick it with cross-validation.

In [4]:
import json

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed, so charts will be skipped. To see them: pip install matplotlib")

from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor

In [5]:
# --- file paths -------------------------------------------------------
# works whether Jupyter was opened in ml/notebooks, in ml, or in the repo root

for base in (Path(".."), Path("."), Path("ml")):
    if (base / "data" / "laptop_price_clean.csv").exists():
        DATA_DIR = base / "data"
        MODEL_DIR = base / "models"
        break
else:
    raise FileNotFoundError(
        "Cannot find data/laptop_price_clean.csv - open this notebook from the ml folder or the ml/notebooks folder."
    )

MODEL_DIR.mkdir(exist_ok=True)

FileNotFoundError: Cannot find data/laptop_price_clean.csv - open this notebook from the ml folder or the ml/notebooks folder.

In [6]:
# --- load the clean data ---------------------------------------------

clean = pd.read_csv(DATA_DIR / "laptop_price_clean.csv")

# True / False -> 1 / 0 so the models can use them
clean["touchscreen"] = clean["touchscreen"].astype(int)
clean["ips"] = clean["ips"].astype(int)

# the columns the models will use
categorical_columns = ["brand", "type", "cpu_brand", "storage_type", "gpu_brand", "os"]
numeric_columns = [
    "inches",
    "touchscreen",
    "ips",
    "resolution_pixels",
    "cpu_speed_ghz",
    "ram_gb",
    "storage_gb",
    "weight_kg",
]

X = clean[categorical_columns + numeric_columns]
y = clean["price_bdt"]

print("rows:", len(X))
print("features:", X.shape[1])

FileNotFoundError: [Errno 2] No such file or directory: 'data\\laptop_price_clean.csv'

## Split into training and testing

In [7]:
# --- split into training and testing --------------------------------
# the model only ever learns from the training part,
# the test part stays hidden until we score it

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("training rows:", len(X_train))
print("testing rows :", len(X_test))

NameError: name 'X' is not defined

## Build the preprocessing

Two jobs before the model sees the data:
1. turn the text columns into numbers (one-hot encoding)
2. scale every column to a similar range (so distance is fair)

In [8]:
# this turns the 6 text columns into numbers:
# one extra 0/1 column per category (brand_Apple, brand_Dell, ...)
preprocessor = ColumnTransformer(
    transformers=[
        (
            "category",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns,
        ),
        ("number", "passthrough", numeric_columns),
    ]
)

NameError: name 'categorical_columns' is not defined

## Choose k

We try every k from 1 to 20. For each one we train 5 times on different slices of the **training** data and average the error. The k with the **lowest MAE** wins.

In [9]:
# --- try k = 1 ... 20 using cross-validation on the training data ----

k_values = list(range(1, 21))
k_mae = []

for k in k_values:
    trial = Pipeline([
        ("preprocessing", preprocessor),
        ("scaling", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=k)),
    ])
    scores = -cross_val_score(trial, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
    k_mae.append(scores.mean())

best_k = k_values[int(np.argmin(k_mae))]
print("best k:", best_k, "  (CV MAE =", round(min(k_mae)), "BDT)")

if plt is not None:
    plt.figure(figsize=(7, 3.5))
    plt.plot(k_values, k_mae, marker="o")
    plt.axvline(best_k, color="red", linestyle="--", label=f"best k = {best_k}")
    plt.xlabel("k (number of neighbours)")
    plt.ylabel("CV MAE (BDT) — lower is better")
    plt.title("KNN: choosing k")
    plt.legend()
    plt.tight_layout()
    plt.show()

NameError: name 'preprocessor' is not defined

## Build the pipeline with the best k

In [10]:
# 3 steps: turn text into numbers -> scale them -> KNN with the best k
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("scaling", StandardScaler()),
    ("model", KNeighborsRegressor(n_neighbors=best_k)),
])

NameError: name 'preprocessor' is not defined

In [11]:
pipeline.fit(X_train, y_train)
print("model trained")

NameError: name 'pipeline' is not defined

## Score it

Three numbers, same meaning everywhere in this project:

- **MAE** — average miss in taka ("on average we are off by X BDT")
- **RMSE** — like MAE, but big mistakes are punished harder
- **R²** — how much of the price spread the model explains (1.0 = perfect, 0 = useless)

We score twice: on the held-out **test set**, and with **5-fold cross-validation** (more reliable — the data is shuffled and scored 5 different ways).

In [ ]:
# --- score on the test set -------------------------------------------

y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
r2_train = pipeline.score(X_train, y_train)

print("R² on training data :", round(r2_train, 3))
print("R² on test data     :", round(r2, 3))
print("MAE                 :", round(mae), "BDT")
print("RMSE                :", round(rmse), "BDT")

## Cross-validation

Shuffle the data, train 5 times on different slices, average the scores.

In [ ]:
# --- 5-fold cross-validation on the full data -----------------------
# splits the data 5 ways, trains 5 times, averages the scores

cv_mae = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_mean_absolute_error")
cv_rmse = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_root_mean_squared_error")
cv_r2 = cross_val_score(pipeline, X, y, cv=5, scoring="r2")

print("CV MAE :", round(cv_mae.mean()), "±", round(cv_mae.std()))
print("CV RMSE:", round(cv_rmse.mean()), "±", round(cv_rmse.std()))
print("CV R²  :", round(cv_r2.mean(), 3), "±", round(cv_r2.std(), 3))

### Who are the neighbours? (this is the whole idea of KNN)

For one test laptop we show its nearest laptops from the training data. The prediction is just the **average of their prices**.

In [ ]:
# --- show the nearest neighbours of one test laptop ------------------

one_laptop = X_test.iloc[[0]]

# preprocessing + scaling only (everything except the final model)
prepare = pipeline[:-1]
distances, positions = pipeline.named_steps["model"].kneighbors(prepare.transform(one_laptop))

neighbours = X_train.iloc[positions[0]][["brand", "type", "cpu_brand", "ram_gb", "storage_gb", "storage_type"]].copy()
neighbours["price_bdt"] = y_train.iloc[positions[0]].values

print("the laptop we want to price:")
print(one_laptop[["brand", "type", "cpu_brand", "ram_gb", "storage_gb", "storage_type"]].to_string(index=False))
print()
print(f"its {best_k} nearest laptops in the training data:")
print(neighbours.to_string(index=False))
print()
print("average of their prices :", round(neighbours["price_bdt"].mean()), "BDT   <- the KNN prediction")
print("model.predict says      :", round(pipeline.predict(one_laptop)[0]), "BDT")
print("real price              :", y_test.iloc[0], "BDT")

## Try it on 3 real laptops

In [ ]:
# --- try it on 3 real laptops from the test set ---------------------

sample = X_test.head(3)
sample_predicted = pipeline.predict(sample)

for position in range(3):
    real_price = y_test.iloc[position]
    predicted_price = round(sample_predicted[position])
    print("real:", real_price, "BDT    predicted:", predicted_price, "BDT")

## Save the scores

Notebook `08_compare_models.ipynb` reads these files.

In [ ]:
# --- save the scores (notebook 08 reads these files) ----------------

metrics = {
    "model": "KNN",
    "test_mae": round(mae, 2),
    "test_rmse": round(rmse, 2),
    "test_r2": round(r2, 4),
    "cv_mae": round(cv_mae.mean(), 2),
    "cv_rmse": round(cv_rmse.mean(), 2),
    "cv_r2": round(cv_r2.mean(), 4),
}

metrics_path = MODEL_DIR / "metrics_knn.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("saved:", metrics_path)

**Next notebook:** `07_gradient_boosting.ipynb`